# CEUX-3 UDAG project grid

The UDAG CEUX-3 grid is under registration at the time of the creation of this notebook

https://github.com/WCRP-CORDEX/cordex-cmip6-cv/issues/429#issuecomment-4412746383


|                        | CEU-3          | MEU-3      | CEUX-3                  |
|------------------------|----------------|------------|-------------------------|
| Domain                 | Central Europe | Mid-Europe | Central Europe Extended |
| Grid spacing           | 0.0275°        | 0.0275°    | 0.0275°                 |
| Rotated pole longitude | -162.0         |  -162.0    |  -162.0                 |
| Rotated pole latitude  | 39.25          | 39.25      | 39.25                   |
| xfirst (rlon start)    | -9.30375       | -12.3      |  -12.30125              |
| yfirst (rlat start)    | -5.07375       | -6.9       | -6.91625                |
| xsize                  | 345            | 501        | 502                     |
| ysize                  | 385            | 501        | 502                     |

During personal communication, it turned out that the CEUX-3 domain is supposed to either be identical to MEU-3 (in terms of values) or contain it entirely. Official definitions will be made by CORDEX-CMIP7 task teams in the future. Until then, we are free to choose the exact values. We decided to set the definitions in accordance with the metadata of UDAG .nc files, which will be published on ESGF server and can be found on DWD HPC (rcl) at

/hpc/uwork/icon-ku/cmor_published/CPRCM/CEUX-3/

For any arbitrary UDAG output file, the command 'ncdump -h *.nc' shows:

````
dimensions:
        ...
        rlat = 501 ;
        rlon = 501 ;
        bnds = 2 ;
        ...
variables:
        ...
        int crs ;
                crs:grid_mapping_name = "rotated_latitude_longitude" ;
                crs:grid_north_pole_latitude = 39.25 ;
                crs:grid_north_pole_longitude = -162. ;
                crs:north_pole_grid_longitude = 0. ;
                crs:earth_radius = 6371229. ;
        ...
// global attributes:
                ...
                :domain = "Central Europe Extended" ;
                :domain_id = "CEUX-3" ;
                ...
````

and 'cdo sinfo *.nc' shows:

````
File format : NetCDF4 classic zip
    -1 : Institut Source   T Steptype Levels Num    Points Num Dtype : Parameter ID
     1 : unknown  ICON     v instant       1   1    251001   1  F32z : -1            
   Grid coordinates :
     1 : curvilinear              : points=251001 (501x501)
                              lon : -4.080024 to 20.68554 degrees_east
                              lat : 42.46392 to 57.6 degrees_north
                        available : cellbounds
                          mapping : rotated_latitude_longitude
                             rlon : -12.3 to 1.45 by 0.0275 degrees
                             rlat : -6.9 to 6.85 by 0.0275 degrees
                        available : cellbounds
   Vertical coordinates :
     1 : surface                  : levels=1
   Time coordinate :
                             time : 8760 steps
````

Our definition is consequently:

|                        | CEU-3          | CEUX-3                  |
|------------------------|----------------|-------------------------|
| Domain                 | Central Europe | Central Europe Extended |
| Grid spacing           | 0.0275°        | 0.0275°                 | 
| Rotated pole longitude | -162.0         | -162.0                  |
| Rotated pole latitude  | 39.25          | 39.25                   | 
| xfirst (rlon start)    | -9.30375       | -12.3                   |  
| yfirst (rlat start)    | -5.07375       | -6.9                    | 
| xsize                  | 345            | 501                     | 
| ysize                  | 385            | 501                     | 


# Earth radius.

Without a clear reference, the CORDEX earth radius is taken from:

Reference: https://wcrp-cordex.github.io/archive-specifications/CORDEX-CMIP6_archiving_specifications_DD/#131-rotated-pole-coordinate-system

````
char crs ;
    crs:grid_mapping_name = "rotated_latitude_longitude" ;
    crs:grid_north_pole_latitude = 39.25 ;
    crs:grid_north_pole_longitude = -162. ;
    crs:earth_radius = 6371229. ;
````

## Converting grid_north_pole_longitude (CF-conform) to lon_0 (PROJ conform)

In [ ]:
grid_north_pole_longitude = -162.0

lon_0 = grid_north_pole_longitude + 180
if lon_0 > 180:
    lon_0 -= 360
print(lon_0)

## Grid information

In [ ]:
grids = {
    "CEU-3": {
        "domain": "Central Europe",
        "grid_spacing": 0.0275,
        "rotated_pole_longitude": -162.0,
        "rotated_pole_latitude": 39.25,
        "xfirst": -9.30375,
        "yfirst": -5.07375,
        "xsize": 345,
        "ysize": 385,
    },
    "CEUX-3": {
        "domain": "Central Europe Extended",
        "grid_spacing": 0.0275,
        "rotated_pole_longitude": -162.0,
        "rotated_pole_latitude": 39.25,
        "xfirst": -12.3,
        "yfirst": -6.9,
        "xsize": 501,
        "ysize": 501,
    },
}

## Generate pyresample area definition

In [ ]:
import pyproj
from pyresample import get_area_def

def generate_pyku_configuration(grid):
    proj_string = (
        "+proj=ob_tran +o_proj=longlat +R=6371229. +o_lat_p=39.25 +o_lon_p=0 +lon_0=18"
    )

    x_size = grids[grid]['xsize']
    y_size = grids[grid]['ysize']

    x_ll, y_ll = (
        grids[grid]['xfirst'] - grids[grid]['grid_spacing']/2,
        grids[grid]['yfirst'] - grids[grid]['grid_spacing']/2
    )
    
    x_ur, y_ur = (
        grids[grid]['xfirst']- grids[grid]['grid_spacing']/2 + grids[grid]['grid_spacing'] * grids[grid]['xsize'],
        grids[grid]['yfirst']- grids[grid]['grid_spacing']/2 + grids[grid]['grid_spacing'] * grids[grid]['ysize']
    )

    # There is a small error on the last decimale, likely due to float representation conversion
    x_ll, y_ll = round(x_ll, 10), round(y_ll, 10)
    x_ur, y_ur = round(x_ur, 10), round(y_ur, 10)
    
    area_id = grid
    
    area_name = (
        f"{grid} rotated longitude latitude grid in accordance with future CORDEX domains "
        "https://github.com/WCRP-CORDEX/cordex-cmip6-cv/issues/429#issuecomment-4412746383"
    )
    
    proj_id = None
    proj4_args = proj_string

    print(f"""\n>>>>>areas.yaml""")
    
    area_extent = (x_ll, y_ll, x_ur, y_ur)
    area_def = get_area_def(
        area_id,
        area_name,
        proj_id,
        proj4_args,
        x_size,
        y_size,
        area_extent
    )
    
    print(area_def.dump())

    print(f"""\n>>>>>areas.yaml""")
    print(f"""
{grid}:
    crs_name: rotated_pole
    crs_data:
        grid_mapping_name: rotated_latitude_longitude
        grid_north_pole_latitude: 39.25
        grid_north_pole_longitude: -162.0
        earth_radius: 6371229.
        long_name: coordinates of the rotated North Pole
    y_coordinate: 'rlat'
    x_coordinate: 'rlon'
    CORDEX_domain: '{grid}'
    proj4: '{area_def.proj_str}'
    crs_wkt: '{area_def.crs_wkt}'
""")

## Generate CF-conform configuration

In [ ]:
for grid in ['CEU-3', 'CEUX-3']:
    generate_pyku_configuration(grid)